In [1]:
import pandas as pd

# 💡 수정된 부분: F드라이브의 다운로드 폴더로 절대 경로 지정
file_path = r'F:\Download\Step3_통합_보험데이터_최종_1개월_월환산완료.csv'

try:
    print("⏳ 데이터를 불러오는 중입니다...")
    # 한글 깨짐 방지를 위해 utf-8 시도 후 실패 시 cp949 적용
    try:
        df = pd.read_csv(file_path)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='cp949')
        
    print("✅ 데이터 로드 성공!\n")
    
    # 2. 보장내역 컬럼 필터링
    # 기본 정보(국가, 상품플랜, 가입나이, 성별, 최종보험료)를 제외한 나머지 컬럼을 보장항목으로 간주
    basic_cols = ['국가', '상품플랜', '가입나이', '성별', '최종보험료']
    coverage_cols = [col for col in df.columns if col not in basic_cols]
    
    # 3. 보장항목별 요약 통계 계산
    summary_list = []
    
    for col in coverage_cols:
        # 데이터를 숫자로 강제 변환 (문자열이 섞여 있을 경우 대비)
        numeric_series = pd.to_numeric(df[col], errors='coerce').fillna(0)
        
        # 보장금액이 0보다 큰 플랜만 추출 (실제 해당 보장을 제공하는 경우)
        active_coverage = numeric_series[numeric_series > 0]
        
        if len(active_coverage) > 0:
            avg_amt = active_coverage.mean()
            max_amt = active_coverage.max()
            min_amt = active_coverage.min()
            count = len(active_coverage)
        else:
            avg_amt = max_amt = min_amt = count = 0
            
        summary_list.append({
            '보장항목': col,
            '제공_플랜수': count,
            '최소보장비용': min_amt,
            '평균보장비용': avg_amt,
            '최대보장비용': max_amt
        })
        
    # 데이터프레임으로 변환
    summary_df = pd.DataFrame(summary_list)
    
    # 4. 보기 좋게 금액 포맷팅 (세 자리 콤마 추가 및 소수점 제거)
    for col in ['최소보장비용', '평균보장비용', '최대보장비용']:
        summary_df[col] = summary_df[col].apply(lambda x: f"{int(x):,}")
        
    # '제공_플랜수'를 기준으로 내림차순 정렬 (가장 많이 제공되는 보장부터 확인)
    summary_df = summary_df.sort_values(by='제공_플랜수', ascending=False).reset_index(drop=True)
    
    print("📊 [보장내역별 보장비용 요약]")
    print("-" * 90)
    print(summary_df.to_string(index=False))
    print("-" * 90)
    
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. \n👉 확인 경로: {file_path}\n파일명이나 폴더 위치가 정확한지 확인해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 데이터를 불러오는 중입니다...
✅ 데이터 로드 성공!

📊 [보장내역별 보장비용 요약]
------------------------------------------------------------------------------------------
                보장항목  제공_플랜수     최소보장비용     평균보장비용      최대보장비용
          사망및장해_상해사망      56 10,000,000 90,000,000 200,000,000
        사망및장해_상해후유장해      56 10,000,000 90,000,000 200,000,000
사망및장해_질병사망및80%이상후유장해      56  5,000,000 18,750,000  50,000,000
           일상손해_배상책임      42  5,000,000 15,000,000  30,000,000
    일상손해_휴대품손해(분실제외)      42    200,000    466,666   1,000,000
       3대비급여_MRI/MRA      14  3,000,000  3,000,000   3,000,000
          통원의료비_상해외래      14    250,000    250,000     250,000
  특수리스크_해외상해의료비(USD)      14 20,250,000 20,250,000  20,250,000
       특수리스크_항공기지연비용      14    100,000    100,000     100,000
         특수리스크_구원자비용      14 10,000,000 10,000,000  10,000,000
        통원의료비_질병처방조제      14     50,000     50,000      50,000
          통원의료비_질병외래      14    250,000    250,000     250,000
        통원의료비_상해처방조제      14     50,00

In [1]:
import pandas as pd
import numpy as np

# 1. 파일 경로 설정
input_file_path = r'F:\Download\Step3_통합_보험데이터_최종_1개월_월환산완료.csv'
output_file_path = r'F:\Download\Step3_Coverage_Cost_Weight_Table.xlsx'

try:
    print("⏳ 데이터를 불러오고 있습니다...")
    try:
        df = pd.read_csv(input_file_path)
    except UnicodeDecodeError:
        df = pd.read_csv(input_file_path, encoding='cp949')
        
    # 2. 보장내역 컬럼 필터링
    basic_cols = ['국가', '상품플랜', '가입나이', '성별', '최종보험료']
    coverage_cols = [col for col in df.columns if col not in basic_cols]
    
    summary_list = []
    
    # 3. 보장항목별 평균 금액 계산
    print("⚙️ 담보별 평균 보장비용 및 가중치를 계산합니다...")
    for col in coverage_cols:
        numeric_series = pd.to_numeric(df[col], errors='coerce').fillna(0)
        active_coverage = numeric_series[numeric_series > 0]
        
        if len(active_coverage) > 0:
            avg_amt = active_coverage.mean()
        else:
            avg_amt = 0
            
        summary_list.append({
            'Coverage Item': col,
            'Average_Cost': avg_amt
        })
        
    weight_df = pd.DataFrame(summary_list)
    
    # 평균비용이 0인 담보 제외
    weight_df = weight_df[weight_df['Average_Cost'] > 0].copy()
    
    # 4. 가중치 정규화 로직 (로그 변환 + Min-Max Scaling)
    # 금액 편차가 워낙 크기 때문에(수억 원 vs 수십만 원) 로그를 씌워 편차를 줄여줍니다.
    weight_df['Log_Cost'] = np.log1p(weight_df['Average_Cost'])
    
    max_log = weight_df['Log_Cost'].max()
    min_log = weight_df['Log_Cost'].min()
    
    # 0 ~ 1 사이의 값으로 정규화 (Cost Weight)
    # 가장 금액이 작은 보장이 0.1, 가장 큰 보장이 1.0이 되도록 최소값 0.1을 보정해줍니다.
    weight_df['Cost Weight (0-1)'] = (
        (weight_df['Log_Cost'] - min_log) / (max_log - min_log) * 0.9 + 0.1
    ).round(4)
    
    # 보기 좋게 정렬 및 포맷팅
    weight_df = weight_df.sort_values(by='Cost Weight (0-1)', ascending=False).reset_index(drop=True)
    weight_df['Average_Cost_Formatted'] = weight_df['Average_Cost'].apply(lambda x: f"{int(x):,}")
    
    # 최종 출력용 데이터 (로그값은 숨김)
    final_display = weight_df[['Coverage Item', 'Average_Cost_Formatted', 'Cost Weight (0-1)']]
    
    print("\n✅ [보장내역별 비용 가중치(Cost Weight) 산출 결과]")
    print("-" * 75)
    print(final_display.to_string(index=False))
    print("-" * 75)
    
    # 5. 결과를 추천 모델에 적용할 수 있도록 엑셀로 저장
    # 사용할 핵심 컬럼만 저장 (이 파일이 WeightedScoringRecommender의 df_cost_weights 로 들어갑니다)
    export_df = weight_df[['Coverage Item', 'Average_Cost', 'Cost Weight (0-1)']]
    export_df.to_excel(output_file_path, index=False, engine='openpyxl')
    
    print(f"\n💾 추천 엔진용 가중치 파일이 저장되었습니다:\n   👉 {output_file_path}")

except FileNotFoundError:
    print(f"❌ '{input_file_path}' 파일을 찾을 수 없습니다.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 데이터를 불러오고 있습니다...
⚙️ 담보별 평균 보장비용 및 가중치를 계산합니다...

✅ [보장내역별 비용 가중치(Cost Weight) 산출 결과]
---------------------------------------------------------------------------
       Coverage Item Average_Cost_Formatted  Cost Weight (0-1)
          사망및장해_상해사망             90,000,000             1.0000
        사망및장해_상해후유장해             90,000,000             1.0000
        입원의료비_상해질병통합             50,000,000             0.9294
  특수리스크_해외상해의료비(USD)             20,250,000             0.8209
사망및장해_질병사망및80%이상후유장해             18,750,000             0.8117
           일상손해_배상책임             15,000,000             0.7849
  특수리스크_해외질병의료비(USD)             13,500,000             0.7722
      입원의료비_상해급여(국내)             10,000,000             0.7362
      입원의료비_질병급여(국내)             10,000,000             0.7362
         특수리스크_구원자비용             10,000,000             0.7362
          3대비급여_통합한도              9,000,000             0.7235
          입원의료비_상해입원              5,000,000             0.6529
          입원의료비_질